## 导入模块并初始化


In [ ]:
from pathlib import Path
import pandas as pd
from market_data import create_manager, today_str, load_tushare_token

In [ ]:
# Tushare Token
TUSHARE_TOKEN = load_tushare_token()

# 数据库路径：直接放在项目目录下
DB_PATH = "data/db/market_data.db"

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20150101",
    default_exchange="SSE",
)

print("数据库位置：", Path(DB_PATH).resolve())

## 2. 初始化基础数据
项目开始时先跑一次，用来获取：
场内基金基础信息
交易日历

In [ ]:
manager.initialize_basic_data(
    fund_market="E",      # 场内基金
    fund_status="L",      # 上市状态
    cal_start_date="20150101",
    cal_end_date=today_str(),
    exchange="SSE",
)

## 3. 查看当前本地资产列表

In [ ]:
instruments = manager.store.get_instruments(listed_only=True)
print(instruments.shape)
instruments.head()

## 4. 查看本地已收录的 ts_code 列表

In [ ]:
ts_codes = manager.store.get_ts_codes(listed_only=True)
len(ts_codes), ts_codes[:10]

5. 更新单个基金/ETF 日线数据

例如“515100.SH”

In [ ]:
df_515100 = manager.update_one_daily_price(
    ts_code="515100.SH",
    start_date="20180101",
    end_date=today_str(),
)
df_515100.tail()

## 6. 批量更新指定资产池

In [ ]:
watchlist = [
    "510300.SH",  # 沪深300ETF
    "511010.SH",  # 国债ETF
    "518880.SH",  # 黄金ETF
]

summary = manager.update_daily_prices(
    ts_codes=watchlist,
    start_date="20150101",
    end_date=today_str(),
)

summary

## 7. 对全库做增量更新

In [ ]:
summary_all = manager.update_daily_prices()
summary_all

## 8. 只刷新最近一段时间的数据
这个功能适合日常维护。
比如你已经有历史全量数据了，之后每天只想回补最近 30 天或 60 天。

In [ ]:
recent_summary = manager.refresh_recent_daily_prices(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"],
    lookback_days=30,
    end_date=today_str(),
)
recent_summary

如果想对全体上市基金刷新最近 30 天：

In [ ]:
recent_summary_all = manager.refresh_recent_daily_prices(
    lookback_days=30,
    end_date=today_str(),
)
recent_summary_all

## 9. 检查本地价格数据覆盖范围

In [ ]:
coverage = manager.check_price_coverage(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"]
)
coverage

检查全库：

In [ ]:
coverage_all = manager.check_price_coverage()
coverage_all.head(20)

筛选出还没有数据的标的：

In [ ]:
coverage_all[coverage_all["has_data"] == 0].head(20)

## 10. 检查数据是否已经更新到最新交易日

In [ ]:
status = manager.check_latest_price_status(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"],
    exchange="SSE",
)
status

筛选出未更新到最新交易日的资产：

In [ ]:
status[status["is_up_to_date"] == 0]

## 11. 读取本地已存储的日线数据
 读取单个标的

In [ ]:
prices_510300 = manager.store.get_daily_prices(
    ts_codes=["510300.SH"],
    start_date="20230101",
    end_date=today_str(),
)
prices_510300.tail()

读取多个标的

In [ ]:
prices = manager.store.get_daily_prices(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"],
    start_date="20230101",
    end_date=today_str(),
)
prices.head()

## 12. 透视查看单个字段（比如 close）

In [ ]:
prices = manager.store.get_daily_prices(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"],
    start_date="20230101",
    end_date=today_str(),
)

close_df = prices.pivot(index="trade_date", columns="ts_code", values="close").sort_index()
close_df.tail()

成交额矩阵

In [ ]:
amount_df = prices.pivot(index="trade_date", columns="ts_code", values="amount").sort_index()
amount_df.tail()

## 13. 查看交易日历

In [ ]:
trade_cal = manager.store.get_trade_calendar(
    exchange="SSE",
    start_date="20240101",
    end_date=today_str(),
)
trade_cal.tail()

只看开市日

In [ ]:
open_dates = manager.store.get_open_trade_dates(
    exchange="SSE",
    start_date="20240101",
    end_date=today_str(),
)
open_dates[-10:]

## 14. 查看某个标的的数据起止日期

In [ ]:
manager.store.get_price_date_range("510300.SH")

查看某个标的最新交易日：

In [ ]:
manager.store.get_latest_trade_date("510300.SH")

## 15. 查看哪些标的还没有本地日线数据

In [ ]:
missing_codes = manager.store.get_missing_ts_codes(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"]
)
missing_codes

查看全库

In [ ]:
missing_all = manager.store.get_missing_ts_codes()
len(missing_all)

## 16. 查看哪些标的数据为空或滞后
比如要所有标的至少更新到某天：

In [ ]:
stale_codes = manager.store.get_empty_or_stale_ts_codes(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"],
    expected_latest_trade_date="20260320",
)
stale_codes

## 17.查看更新日志
查看最近的

In [ ]:
logs = manager.store.get_update_log(limit=20)
logs

只看日线更新日志：

In [ ]:
logs_daily = manager.store.get_update_log(
    table_name="daily_prices",
    limit=20,
)
logs_daily

只看某个标的

In [ ]:
logs_510300 = manager.store.get_update_log(
    table_name="daily_prices",
    ts_code="510300.SH",
    limit=20,
)
logs_510300

## 18. 最常用的日常维护流程示例

以后作为 notebook 的“每日更新入口”。

说明：
- 本 cell 可以独立运行
- 会显示当前 `daily_prices` 的默认口径（raw/qfq/hfq）
- `refresh_recent_daily_prices` 在新版本 `market_data.py` 下会同时维护 `daily_prices_raw` 和物化后的 `daily_prices`

In [ ]:
from pathlib import Path
import pandas as pd
from market_data import create_manager, today_str, load_tushare_token

# ===== 常用参数（可独立运行） =====
TUSHARE_TOKEN = load_tushare_token()
DB_PATH = "data/db/market_data.db"
EXCHANGE = "SSE"

watchlist = [
    "510300.SH",  # 沪深300ETF
    "510500.SH",  # 中证500ETF
    "510170.SH",  # 商品/大宗相关ETF
    "159915.SZ",  # 创业板ETF
    "511090.SH",  # 债券ETF
    "511010.SH",  # 国债ETF
    "518880.SH",  # 黄金ETF
    "159981.SZ",  # 能源ETF
    "159985.SZ",  # 豆粕ETF
    "501018.SH",  # 原油LOF
    "515100.SH",  # 红利低波ETF
]

# 如有混合资产池（股票 + ETF/LOF），可以在这里显式指定
# 例：{"000300.SH": "stock", "510300.SH": "fund"}
ASSET_TYPE_MAP = {}

LOOKBACK_DAYS = 30
RECENT_PRICE_START = "20250101"

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20070101",
    default_exchange=EXCHANGE,
)

print("数据库位置：", Path(DB_PATH).resolve())
print("当前默认价格口径：", manager.store.get_default_adjust_type())

# 1. 刷新最近一段时间数据
# 新版 market_data.py 下，这一步会：
# - 先更新 daily_prices_raw
# - 再按当前默认口径（raw/qfq/hfq）自动更新 daily_prices
recent_summary = manager.refresh_recent_daily_prices(
    ts_codes=watchlist,
    lookback_days=LOOKBACK_DAYS,
    end_date=today_str(),
    listed_only=False,
    asset_type_map=ASSET_TYPE_MAP or None,
)

print("\n最近数据刷新结果：")
print(recent_summary)

# 2. 检查最新状态（materialized daily_prices）
status = manager.check_latest_price_status(
    ts_codes=watchlist,
    exchange=EXCHANGE,
    listed_only=False,
)

print("\n最新状态：")
display(status)

# 3. 检查复权状态（raw / factor / materialized）
adj_status = manager.check_adjustment_status(
    ts_codes=watchlist,
    listed_only=False,
)

print("\n复权状态：")
display(adj_status)

# 4. 读取 materialized 价格（供回测/策略直接使用）
prices = manager.store.get_daily_prices(
    ts_codes=watchlist,
    start_date=RECENT_PRICE_START,
    end_date=today_str(),
)

print("\nmaterialized 最近价格数据：")
display(prices.tail(10))

# 5. 如需排查，可顺手看 raw 价格
raw_prices = manager.store.get_raw_daily_prices(
    ts_codes=watchlist,
    start_date=RECENT_PRICE_START,
    end_date=today_str(),
)

print("\nraw 最近价格数据：")
display(raw_prices.tail(10))

## 19. 最常用的项目初始化流程示例

这个 cell 适合项目刚开始时跑一次。

说明：
- 支持基金/ETF 为主，也可扩展到股票
- 可选把 `daily_prices` 一次性迁移到 `qfq/hfq/raw` 默认口径
- 下游策略仍继续直接读取 `daily_prices`

In [ ]:
from pathlib import Path
import pandas as pd
from market_data import create_manager, today_str, load_tushare_token

# ===== 项目初始化参数（可独立运行） =====
TUSHARE_TOKEN = load_tushare_token()
DB_PATH = "data/db/market_data.db"
EXCHANGE = "SSE"

# 基础数据初始化
FUND_MARKET = "E"
FUND_STATUS = "L"
INIT_STOCK_BASICS = False   # 如果资产池里包含股票，建议改成 True
STOCK_EXCHANGE = ""         # "", "SSE", "SZSE", "BSE"
STOCK_LIST_STATUS = "L"

# 历史数据初始化范围
HISTORY_START = "20070101"
HISTORY_END = today_str()

# 初始化后是否把 daily_prices 物化成默认复权口径
ENABLE_ADJUSTMENT_MIGRATION = True
DEFAULT_ADJUST_TYPE = "qfq"   # 可选: "raw" / "qfq" / "hfq"

watchlist = [
    "510300.SH",
    "510500.SH",
    "510170.SH",
    "159915.SZ",
    "511090.SH",
    "511010.SH",
    "518880.SH",
    "159981.SZ",
    "159985.SZ",
    "501018.SH",
    "515100.SH",
]

# 如有混合资产池（股票 + ETF/LOF），可以在这里显式指定
ASSET_TYPE_MAP = {}

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date=HISTORY_START,
    default_exchange=EXCHANGE,
)

print("数据库位置：", Path(DB_PATH).resolve())

# Step 1: 初始化基础数据（基金基础信息 + 交易日历；可选股票基础信息）
manager.initialize_basic_data(
    fund_market=FUND_MARKET,
    fund_status=FUND_STATUS,
    cal_start_date=HISTORY_START,
    cal_end_date=HISTORY_END,
    exchange=EXCHANGE,
    include_stocks=INIT_STOCK_BASICS,
    stock_exchange=STOCK_EXCHANGE,
    stock_list_status=STOCK_LIST_STATUS,
)

# 如需单独刷新股票基础信息，也可手动调用
# manager.update_stock_instruments(exchange=STOCK_EXCHANGE, list_status=STOCK_LIST_STATUS)

# Step 2: 拉取/补齐原始日线，并自动维护 materialized daily_prices
summary = manager.update_daily_prices(
    ts_codes=watchlist,
    start_date=HISTORY_START,
    end_date=HISTORY_END,
    listed_only=False,
    asset_type_map=ASSET_TYPE_MAP or None,
)

print("\n历史数据初始化完成：")
print(summary)

# Step 3: 可选，把 daily_prices 切换到默认复权口径
if ENABLE_ADJUSTMENT_MIGRATION:
    migrate_summary = manager.migrate_to_materialized_adjusted_prices(
        ts_codes=watchlist,
        adjust_type=DEFAULT_ADJUST_TYPE,
        listed_only=False,
        backup_existing_daily=True,
        asset_type_map=ASSET_TYPE_MAP or None,
    )
    print("\n复权迁移/物化完成：")
    print(migrate_summary)
else:
    print("\n跳过复权迁移，当前默认价格口径：", manager.store.get_default_adjust_type())

# Step 4: 检查覆盖范围
coverage = manager.check_price_coverage(ts_codes=watchlist, listed_only=False)
print("\n价格覆盖范围：")
display(coverage)

# Step 5: 检查是否更新到最新交易日
status = manager.check_latest_price_status(ts_codes=watchlist, exchange=EXCHANGE, listed_only=False)
print("\n最新交易日检查：")
display(status)

# Step 6: 检查复权状态
adj_status = manager.check_adjustment_status(ts_codes=watchlist, listed_only=False)
print("\n复权状态：")
display(adj_status)

print("\n当前默认价格口径：", manager.store.get_default_adjust_type())

## 20. 新资产池智能补数（只补缺口，不重复拉取）

适用于：换了一套和旧资产池差异很大的新资产池，但其中部分标的、部分日期可能已经在本地数据库里。

这个 cell 会：
1. 自己初始化 manager（可独立运行）
2. 优先按 `daily_prices_raw` 检查每个标的在指定区间内的本地覆盖情况
3. 按缺失交易日拆成连续区间
4. 只对缺口区间发起拉取，避免重复抓取已经存在的数据
5. 新版 `market_data.py` 下会自动维护 materialized `daily_prices`

In [ ]:
from pathlib import Path
import pandas as pd
from market_data import create_manager, today_str, load_tushare_token

# ========= 新资产池：智能补数（只补缺口，不重复拉取） =========
NEW_WATCHLIST = [
    "510300.SH",
    "513100.SH",
    "518880.SH",
    "511010.SH",
    "510880.SH",
    "162411.SZ",
    "510170.SH",
]

# 如有混合资产池（股票 + ETF/LOF），可以在这里显式指定
ASSET_TYPE_MAP = {}

START_DATE = "20130801"
END_DATE = today_str()
DB_PATH = "data/db/market_data.db"
EXCHANGE = "SSE"

# 可选：如果你怀疑交易日历没更新，可以保持 True
ENSURE_TRADE_CALENDAR = True

# ===== 独立初始化（无需依赖前置 cell） =====
TUSHARE_TOKEN = load_tushare_token()
manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20070101",
    default_exchange=EXCHANGE,
)

print("数据库位置：", Path(DB_PATH).resolve())
print("当前默认价格口径：", manager.store.get_default_adjust_type())

if ENSURE_TRADE_CALENDAR:
    _ = manager.update_trade_calendar(
        exchange=EXCHANGE,
        start_date=START_DATE,
        end_date=END_DATE,
    )

open_dates = manager.store.get_open_trade_dates(
    exchange=EXCHANGE,
    start_date=START_DATE,
    end_date=END_DATE,
)
if len(open_dates) == 0:
    raise ValueError("指定区间内没有拿到开市交易日，请先检查交易日历或日期参数。")

open_dates = [str(x) for x in open_dates]
open_pos = {d: i for i, d in enumerate(open_dates)}

# 本地基础信息（若已有 instruments，可用于排除上市前日期）
inst = manager.store.get_instruments(listed_only=False)
if len(inst) > 0 and "ts_code" in inst.columns:
    inst = inst.copy()
    if "list_date" in inst.columns:
        inst["list_date"] = inst["list_date"].astype(str)
    inst_map = inst.set_index("ts_code").to_dict("index")
else:
    inst_map = {}


def split_missing_ranges(missing_dates, pos_map):
    if not missing_dates:
        return []

    ordered = sorted(missing_dates, key=lambda x: pos_map[x])
    ranges = []
    start = ordered[0]
    prev = ordered[0]

    for d in ordered[1:]:
        if pos_map[d] == pos_map[prev] + 1:
            prev = d
        else:
            ranges.append((start, prev))
            start = d
            prev = d
    ranges.append((start, prev))
    return ranges


coverage_rows = []
fetch_plan_rows = []

for ts_code in NEW_WATCHLIST:
    meta = inst_map.get(ts_code, {})
    list_date = str(meta.get("list_date")) if meta.get("list_date") not in [None, "", "None", "nan"] else None

    effective_start = START_DATE
    if list_date is not None and list_date.isdigit():
        effective_start = max(START_DATE, list_date)

    expected_open_dates = [d for d in open_dates if d >= effective_start]
    expected_open_set = set(expected_open_dates)

    existing_raw = manager.store.get_raw_daily_prices(
        ts_codes=[ts_code],
        start_date=effective_start,
        end_date=END_DATE,
    )
    existing_mat = manager.store.get_daily_prices(
        ts_codes=[ts_code],
        start_date=effective_start,
        end_date=END_DATE,
    )

    if len(existing_raw) > 0:
        existing = existing_raw
        coverage_basis = "raw"
    elif len(existing_mat) > 0:
        # 兼容尚未迁移的旧库：先用 daily_prices 做兜底判断，但建议后续运行第21章
        existing = existing_mat
        coverage_basis = "daily_prices_fallback"
    else:
        existing = pd.DataFrame()
        coverage_basis = "none"

    if len(existing) > 0:
        existing_dates = sorted(existing["trade_date"].astype(str).unique().tolist())
        existing_open_dates = [d for d in existing_dates if d in expected_open_set]
    else:
        existing_dates = []
        existing_open_dates = []

    missing_dates = [d for d in expected_open_dates if d not in set(existing_open_dates)]
    missing_ranges = split_missing_ranges(missing_dates, open_pos)

    coverage_rows.append({
        "ts_code": ts_code,
        "coverage_basis": coverage_basis,
        "list_date": list_date,
        "effective_start": effective_start,
        "request_end": END_DATE,
        "has_any_local_data": int(len(existing_dates) > 0),
        "local_start": existing_dates[0] if existing_dates else None,
        "local_end": existing_dates[-1] if existing_dates else None,
        "expected_open_days": len(expected_open_dates),
        "local_open_days": len(existing_open_dates),
        "missing_open_days": len(missing_dates),
        "missing_ranges": "; ".join([f"{s}~{e}" for s, e in missing_ranges]) if missing_ranges else "",
    })

    for start_date, end_date in missing_ranges:
        fetch_plan_rows.append({
            "ts_code": ts_code,
            "fetch_start": start_date,
            "fetch_end": end_date,
            "asset_type": (ASSET_TYPE_MAP or {}).get(ts_code),
        })

coverage_df = pd.DataFrame(coverage_rows).sort_values("ts_code").reset_index(drop=True)
fetch_plan_df = pd.DataFrame(fetch_plan_rows)

print("=== 本地覆盖检查（优先按 raw 判断） ===")
display(coverage_df)

if len(fetch_plan_df) == 0:
    print("当前新资产池在指定区间内已经完整覆盖，无需新增拉取。")
else:
    print("=== 待拉取缺口区间（只补缺口，不重复拉取） ===")
    display(fetch_plan_df)

    inserted_rows = 0
    updated_codes = set()
    fetch_logs = []

    for row in fetch_plan_df.itertuples(index=False):
        df_new = manager.update_one_daily_price(
            ts_code=row.ts_code,
            start_date=row.fetch_start,
            end_date=row.fetch_end,
            asset_type=row.asset_type,
        )
        n = len(df_new)
        inserted_rows += n
        if n > 0:
            updated_codes.add(row.ts_code)

        fetch_logs.append({
            "ts_code": row.ts_code,
            "fetch_start": row.fetch_start,
            "fetch_end": row.fetch_end,
            "asset_type": row.asset_type,
            "materialized_rows_written": n,
        })

    fetch_log_df = pd.DataFrame(fetch_logs)

    print("=== 本次补数结果 ===")
    display(fetch_log_df)

    print({
        "requested_codes": len(NEW_WATCHLIST),
        "codes_with_gaps": int(fetch_plan_df["ts_code"].nunique()),
        "codes_updated": len(updated_codes),
        "materialized_rows_written": inserted_rows,
    })

    print("=== 补数后再次检查 ===")
    coverage_rows_after = []
    for ts_code in NEW_WATCHLIST:
        meta = inst_map.get(ts_code, {})
        list_date = str(meta.get("list_date")) if meta.get("list_date") not in [None, "", "None", "nan"] else None

        effective_start = START_DATE
        if list_date is not None and list_date.isdigit():
            effective_start = max(START_DATE, list_date)

        expected_open_dates = [d for d in open_dates if d >= effective_start]
        expected_open_set = set(expected_open_dates)

        existing = manager.store.get_raw_daily_prices(
            ts_codes=[ts_code],
            start_date=effective_start,
            end_date=END_DATE,
        )

        if len(existing) > 0:
            existing_dates = sorted(existing["trade_date"].astype(str).unique().tolist())
            existing_open_dates = [d for d in existing_dates if d in expected_open_set]
        else:
            existing_dates = []
            existing_open_dates = []

        missing_dates = [d for d in expected_open_dates if d not in set(existing_open_dates)]

        coverage_rows_after.append({
            "ts_code": ts_code,
            "effective_start": effective_start,
            "request_end": END_DATE,
            "expected_open_days": len(expected_open_dates),
            "local_open_days": len(existing_open_dates),
            "missing_open_days": len(missing_dates),
        })

    coverage_after_df = pd.DataFrame(coverage_rows_after).sort_values("ts_code").reset_index(drop=True)
    display(coverage_after_df)

print("\n=== 当前复权状态 ===")
adj_status = manager.check_adjustment_status(
    ts_codes=NEW_WATCHLIST,
    listed_only=False,
)
display(adj_status)

## 21. 单独检查和处理复权支持

这个 cell 适用于：
1. 检查数据库里当前有哪些资产
2. 判断它们是否已经完成 `raw + 复权因子 + materialized` 支持
3. 自动为缺少复权支持的资产补齐支持
4. 区分：
   - **已有复权支持，且发生过复权事件**
   - **已有复权支持，但因子恒定（通常表示成立以来未发生实际复权事件）**
   - **仍缺少复权因子 / 需要进一步排查权限或代码类型**

说明：
- 这个 cell 会优先保护旧库：如果只有 `daily_prices` 而没有 `daily_prices_raw`，会先做 raw 备份。
- 只有当所有有 raw 数据的资产都能成功建立复权支持时，才会把默认口径切换到 `TARGET_ADJUST_TYPE`。


In [ ]:
from pathlib import Path
import pandas as pd
from market_data import create_manager, today_str, load_tushare_token

# ===== 参数（可独立运行） =====
TUSHARE_TOKEN = load_tushare_token()
DB_PATH = "data/db/market_data.db"
EXCHANGE = "SSE"

TARGET_TS_CODES = None         # None 表示检查库里所有资产；也可手动传 list
TARGET_ADJUST_TYPE = "qfq"     # 可选: "raw" / "qfq" / "hfq"
AUTO_FIX = True                # True: 自动补齐 raw/adj/materialized 支持
BACKUP_LEGACY_DAILY = True     # 对只有 daily_prices、没有 raw 的旧库做保护性备份
ENSURE_TRADE_CALENDAR = True
TRADE_CAL_START = "20070101"
TRADE_CAL_END = today_str()

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date=TRADE_CAL_START,
    default_exchange=EXCHANGE,
)

print("数据库位置：", Path(DB_PATH).resolve())
print("当前默认价格口径：", manager.store.get_default_adjust_type())

if ENSURE_TRADE_CALENDAR:
    _ = manager.update_trade_calendar(
        exchange=EXCHANGE,
        start_date=TRADE_CAL_START,
        end_date=TRADE_CAL_END,
    )

# 1. 收集当前数据库里实际出现过的资产
if TARGET_TS_CODES is None:
    with manager.store.connect() as conn:
        discovered = pd.read_sql_query(
            '''
            SELECT ts_code FROM instruments
            UNION
            SELECT ts_code FROM daily_prices
            UNION
            SELECT ts_code FROM daily_prices_raw
            UNION
            SELECT ts_code FROM asset_adj_factors
            ORDER BY ts_code
            ''',
            conn,
        )
    TARGET_TS_CODES = discovered["ts_code"].dropna().astype(str).tolist()

TARGET_TS_CODES = sorted(set(TARGET_TS_CODES))
print("待检查资产数：", len(TARGET_TS_CODES))

if len(TARGET_TS_CODES) == 0:
    raise ValueError("当前数据库中未发现任何资产。")

asset_type_map = {ts_code: manager.resolve_asset_type(ts_code) for ts_code in TARGET_TS_CODES}

# 2. 对旧库做一次保护性 raw 备份（仅对 raw 缺失但 daily_prices 已有数据的资产）
legacy_only_codes = []
for ts_code in TARGET_TS_CODES:
    raw_start, raw_end = manager.store.get_raw_price_date_range(ts_code)
    mat_start, mat_end = manager.store.get_price_date_range(ts_code)
    if (not raw_start) and mat_start:
        legacy_only_codes.append(ts_code)

backed_up_rows = 0
if BACKUP_LEGACY_DAILY and len(legacy_only_codes) > 0:
    backed_up_rows = manager.store.backup_daily_prices_to_raw(ts_codes=legacy_only_codes)

print("需要从旧 daily_prices 备份到 raw 的资产数：", len(legacy_only_codes))
print("本次备份到 raw 的行数：", backed_up_rows)

# 3. 逐资产检查 / 自动修复
rows = []
fix_logs = []

for ts_code in TARGET_TS_CODES:
    asset_type = asset_type_map.get(ts_code)
    raw_start_before, raw_end_before = manager.store.get_raw_price_date_range(ts_code)
    fac_start_before, fac_end_before = manager.store.get_adj_factor_date_range(ts_code)
    mat_start_before, mat_end_before = manager.store.get_price_date_range(ts_code)

    fetched_factor_rows = 0
    rematerialized_rows = 0
    fix_error = None

    if AUTO_FIX and raw_start_before and raw_end_before:
        try:
            if TARGET_ADJUST_TYPE != "raw":
                fac_before_df = manager.store.get_adj_factors(
                    ts_codes=[ts_code],
                    start_date=raw_start_before,
                    end_date=raw_end_before,
                )
                fac_after_df = manager.ensure_adj_factors_for_asset(
                    ts_code=ts_code,
                    start_date=raw_start_before,
                    end_date=raw_end_before,
                    asset_type=asset_type,
                )
                fetched_factor_rows = max(len(fac_after_df) - len(fac_before_df), 0)
            else:
                fac_after_df = manager.store.get_adj_factors(
                    ts_codes=[ts_code],
                    start_date=raw_start_before,
                    end_date=raw_end_before,
                )

            materialized_df = manager.rematerialize_one_asset(
                ts_code=ts_code,
                adjust_type=TARGET_ADJUST_TYPE,
            )
            rematerialized_rows = len(materialized_df)
        except Exception as e:
            fac_after_df = manager.store.get_adj_factors(
                ts_codes=[ts_code],
                start_date=raw_start_before,
                end_date=raw_end_before,
            )
            fix_error = str(e)
    else:
        fac_after_df = manager.store.get_adj_factors(
            ts_codes=[ts_code],
            start_date=raw_start_before,
            end_date=raw_end_before,
        ) if raw_start_before and raw_end_before else pd.DataFrame()

    raw_start, raw_end = manager.store.get_raw_price_date_range(ts_code)
    fac_start, fac_end = manager.store.get_adj_factor_date_range(ts_code)
    mat_start, mat_end = manager.store.get_price_date_range(ts_code)

    if len(fac_after_df) > 0 and "adj_factor" in fac_after_df.columns:
        fac_numeric = pd.to_numeric(fac_after_df["adj_factor"], errors="coerce").dropna()
        factor_rows = len(fac_numeric)
        factor_unique = fac_numeric.round(12).nunique() if len(fac_numeric) > 0 else 0
        factor_min = float(fac_numeric.min()) if len(fac_numeric) > 0 else None
        factor_max = float(fac_numeric.max()) if len(fac_numeric) > 0 else None
    else:
        factor_rows = 0
        factor_unique = 0
        factor_min = None
        factor_max = None

    if not raw_start and not mat_start:
        support_status = "no_price_data"
    elif raw_start and factor_rows == 0 and TARGET_ADJUST_TYPE != "raw":
        support_status = "factor_missing_or_needs_check"
    elif factor_rows > 0 and factor_unique <= 1:
        support_status = "supported_no_adjust_event"
    elif factor_rows > 0 and factor_unique > 1:
        support_status = "supported_with_adjust_event"
    elif TARGET_ADJUST_TYPE == "raw" and raw_start:
        support_status = "raw_only_mode"
    else:
        support_status = "unknown"

    rows.append({
        "ts_code": ts_code,
        "asset_type": asset_type,
        "support_status": support_status,
        "target_adjust_type": TARGET_ADJUST_TYPE,
        "raw_start": raw_start,
        "raw_end": raw_end,
        "adj_factor_start": fac_start,
        "adj_factor_end": fac_end,
        "materialized_start": mat_start,
        "materialized_end": mat_end,
        "factor_rows": factor_rows,
        "factor_unique": factor_unique,
        "factor_min": factor_min,
        "factor_max": factor_max,
        "fetched_factor_rows": fetched_factor_rows,
        "rematerialized_rows": rematerialized_rows,
        "fix_error": fix_error,
    })

    if AUTO_FIX:
        fix_logs.append({
            "ts_code": ts_code,
            "asset_type": asset_type,
            "fetched_factor_rows": fetched_factor_rows,
            "rematerialized_rows": rematerialized_rows,
            "fix_error": fix_error,
        })

report_df = pd.DataFrame(rows).sort_values(["support_status", "ts_code"]).reset_index(drop=True)

print("\n=== 复权支持检查结果 ===")
display(report_df)

if AUTO_FIX:
    fix_log_df = pd.DataFrame(fix_logs)
    print("\n=== 自动修复日志 ===")
    display(fix_log_df)

# 4. 只有当所有有 raw 数据的资产都已支持目标口径，才切换默认口径
if TARGET_ADJUST_TYPE == "raw":
    manager.store.set_default_adjust_type("raw")
    print("\n默认价格口径已切换为：raw")
else:
    unresolved = report_df[
        report_df["raw_start"].notna() &
        ~report_df["support_status"].isin(["supported_no_adjust_event", "supported_with_adjust_event"])
    ].copy()

    if len(unresolved) == 0:
        manager.store.set_default_adjust_type(TARGET_ADJUST_TYPE)
        print(f"\n默认价格口径已切换为：{TARGET_ADJUST_TYPE}")
    else:
        print(f"\n发现 {len(unresolved)} 个资产尚未完全建立 {TARGET_ADJUST_TYPE} 支持，已保持原默认口径不变。")
        print("建议优先检查这些资产：")
        display(unresolved[[
            "ts_code", "asset_type", "support_status", "raw_start", "raw_end",
            "adj_factor_start", "adj_factor_end", "fix_error"
        ]])

print("\n当前默认价格口径：", manager.store.get_default_adjust_type())